# Checkpoints


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

source_str = "orders"
sink_str = "orders_aggregated"

tn = (
    Tn.source(source_str)
    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "product_ids": sorted(agg_r["product_ids"] + [r["product_id"]])},
                  {"orders": 0, "product_ids": []},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "product_ids": agg_r["product_ids"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r})
    .sink(sink_str)
)

built_tn = Tn.build(tn)



In [11]:
import random

class OrderGenerator:
    def __init__(self):
        self.order_id_int = 0
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = 1

    def generate(self):
        m = {
            "key": self.order_id_int,
            "value": {"id": self.order_id_int,
                      "product_id": random.randint(0, 100 - 1),
                      "customer_id": random.randint(0, 10 - 1),
                      "ts": self.ts_int},
        }
        #
        self.order_id_int += 1
        #
        self.ts_int += self.ts_step_int
        #
        return m

#

gen = OrderGenerator()
for _ in range(3):
    print(gen.generate())


{'key': 0, 'value': {'id': 0, 'product_id': 68, 'customer_id': 2, 'ts': 0}}
{'key': 1, 'value': {'id': 1, 'product_id': 16, 'customer_id': 8, 'ts': 1}}
{'key': 2, 'value': {'id': 2, 'product_id': 25, 'customer_id': 5, 'ts': 2}}


In [ ]:
built_tn.reset()
gen = OrderGenerator()
source_m_list = []
sink_m_list = []
for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

source_key_int_value_dict_dict = {}
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["key"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


In [ ]:
import cloudpickle

#

gen = OrderGenerator()


#

built_tn.reset()
source_m_list = []
sink_m_list = []
for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

x = cloudpickle.dumps(built_tn._evaluator)

# print(built_tn.latest())

built_tn.reset()

# print(built_tn.latest())

y = cloudpickle.loads(x)

built_tn._evaluator = y

# print(built_tn.latest())

#

for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

source_key_int_value_dict_dict = {}
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["key"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


In [65]:
import sys
sys.path.insert(1, "..")

import kafi.streams.streams
import importlib
importlib.reload(kafi.streams.streams)

from kafi.kafka.cluster.cluster import Cluster
from kafi.streams.streams import Streams

c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

source_str = "orders"
sink_str = "orders_aggregated"

tn = (
    Streams.source(c, source_str)
    
    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "order_ids": sorted(agg_r["order_ids"] + [r["id"]]),
                                    "product_ids": sorted(agg_r["product_ids"] + [r["product_id"]])},
                  {"orders": 0, "order_ids": [], "product_ids": []},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "order_ids": agg_r["order_ids"],
                                     "product_ids": agg_r["product_ids"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r}).peek("sink")
    .sink(c, sink_str)
)

built_tn = Streams.build(tn)



In [66]:
from kafi.helpers import get_millis

orders_int = 1000

built_tn.reset()

checkpoint_str = "checkpoint"
g = f"group_{get_millis()}"

c.recreate(source_str)
c.recreate(sink_str)
c.recreate(checkpoint_str)

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, checkpoint_interval=0.01, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)
gen = OrderGenerator()

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



'orders'

Checkpoint consumer group offsets for topic checkpoint: {}
(['checkpoint'], 'group_1785150917408_checkpoint')
source_str_offsets_dict_dict: None
Source consumer group offsets for topic orders: {}
(['orders'], 'group_1785150917408')
sink: {'key': 5, 'value': {'customer_id': 5, 'orders': 107, 'order_ids': [0, 8, 16, 37, 39, 48, 54, 55, 64, 73, 74, 81, 84, 120, 138, 143, 149, 150, 153, 182, 190, 192, 201, 209, 211, 212, 218, 242, 258, 260, 265, 273, 286, 301, 307, 319, 323, 324, 325, 336, 361, 368, 369, 373, 380, 417, 439, 448, 471, 475, 478, 485, 494, 498, 524, 526, 534, 537, 543, 550, 554, 555, 577, 584, 586, 593, 601, 633, 634, 644, 647, 654, 662, 666, 669, 686, 692, 702, 722, 725, 736, 739, 756, 759, 765, 773, 786, 795, 796, 803, 804, 813, 817, 859, 860, 887, 889, 891, 912, 914, 921, 932, 933, 963, 986, 990, 993], 'product_ids': [3, 4, 6, 6, 7, 11, 13, 14, 15, 17, 18, 18, 18, 19, 19, 21, 22, 24, 24, 25, 26, 27, 27, 28, 28, 28, 28, 29, 29, 30, 31, 32, 33, 33, 34, 35, 35, 35, 37, 38, 39

In [67]:
await stop_fun()
await Streams.tasks()

Safely stopping Streams...
...done.


[]

In [68]:
built_tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



{'orders': 1000}
{'orders_aggregated': 10}
{'checkpoint': 31}


'orders'

Checkpoint consumer group offsets for topic checkpoint: {}
(['checkpoint'], 'group_1785150917408_checkpoint')
Loading checkpoint...
source_str_offsets_dict_dict (2): {'orders': {0: 1000}}
...loading checkpoint done.
source_str_offsets_dict_dict: {'orders': {0: 1000}}
Source consumer group offsets for topic orders: {0: 1000}
Source consumer group offsets for topic orders overridden by checkpoint offsets: {0: 1000}
(['orders'], 'group_1785150917408')
sink: {'key': 3, 'value': {'customer_id': 3, 'orders': 189, 'order_ids': [1, 44, 50, 70, 87, 90, 98, 100, 115, 118, 157, 166, 181, 187, 227, 230, 234, 240, 253, 255, 264, 266, 269, 270, 284, 289, 297, 302, 312, 313, 314, 320, 348, 360, 363, 367, 382, 393, 396, 407, 424, 425, 453, 455, 457, 468, 480, 496, 503, 510, 538, 549, 562, 565, 572, 574, 581, 594, 603, 623, 631, 649, 656, 661, 665, 671, 673, 684, 687, 708, 742, 751, 784, 798, 801, 805, 819, 820, 835, 841, 855, 867, 910, 941, 942, 952, 966, 970, 971, 982, 1000, 1009, 1010, 1018, 1020, 1

In [69]:
await stop_fun()
await Streams.tasks()

Safely stopping Streams...
...done.


[]

In [70]:
built_tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()


{'orders': 2000}
{'orders_aggregated': 20}
{'checkpoint': 87}


'orders'

Checkpoint consumer group offsets for topic checkpoint: {0: 31}
(['checkpoint'], 'group_1785150917408_checkpoint')
Loading checkpoint...
source_str_offsets_dict_dict (2): {'orders': {0: 2000}}
...loading checkpoint done.
source_str_offsets_dict_dict: {'orders': {0: 2000}}
Source consumer group offsets for topic orders: {0: 2000}
Source consumer group offsets for topic orders overridden by checkpoint offsets: {0: 2000}
(['orders'], 'group_1785150917408')
sink: {'key': 7, 'value': {'customer_id': 7, 'orders': 311, 'order_ids': [4, 12, 25, 33, 35, 49, 52, 58, 60, 67, 77, 92, 93, 109, 128, 130, 137, 146, 152, 154, 194, 219, 228, 231, 237, 244, 271, 275, 276, 285, 310, 329, 332, 343, 349, 358, 366, 372, 385, 386, 389, 406, 431, 432, 442, 445, 450, 461, 463, 493, 527, 547, 567, 604, 608, 610, 616, 618, 621, 638, 645, 648, 677, 694, 701, 705, 713, 715, 724, 730, 741, 747, 772, 778, 779, 800, 802, 821, 822, 824, 829, 832, 849, 854, 865, 894, 895, 906, 913, 918, 919, 929, 946, 976, 1001, 1002, 

In [71]:
await stop_fun()
await Streams.tasks()

Safely stopping Streams...
...done.


[]

In [72]:
source_key_int_value_dict_dict = {}
source_m_list = c.cat(source_str)
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_order_id_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("order_ids", [])
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "order_ids": sorted(agg_order_id_int_list + [order_id_int]),
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
sink_m_list = c.cat(sink_str)
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["value"]["customer_id"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


(['orders'], '1785150966343')
Read: 1000
Read: 2000
Read: 3000
(['orders_aggregated'], '1785150971633')
{5: {'customer_id': 5, 'orders': 323, 'order_ids': [0, 8, 16, 37, 39, 48, 54, 55, 64, 73, 74, 81, 84, 120, 138, 143, 149, 150, 153, 182, 190, 192, 201, 209, 211, 212, 218, 242, 258, 260, 265, 273, 286, 301, 307, 319, 323, 324, 325, 336, 361, 368, 369, 373, 380, 417, 439, 448, 471, 475, 478, 485, 494, 498, 524, 526, 534, 537, 543, 550, 554, 555, 577, 584, 586, 593, 601, 633, 634, 644, 647, 654, 662, 666, 669, 686, 692, 702, 722, 725, 736, 739, 756, 759, 765, 773, 786, 795, 796, 803, 804, 813, 817, 859, 860, 887, 889, 891, 912, 914, 921, 932, 933, 963, 986, 990, 993, 1027, 1028, 1029, 1031, 1032, 1033, 1036, 1065, 1066, 1068, 1091, 1110, 1113, 1120, 1141, 1161, 1163, 1167, 1168, 1170, 1174, 1175, 1188, 1203, 1205, 1221, 1231, 1248, 1255, 1267, 1289, 1292, 1293, 1312, 1336, 1340, 1341, 1355, 1357, 1385, 1402, 1410, 1420, 1430, 1431, 1436, 1439, 1441, 1449, 1467, 1479, 1498, 1510, 1540, 